In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

# Modelos
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier

# Métricas e Exportação
from sklearn.metrics import accuracy_score, classification_report
import joblib
import pickle

df = pd.read_csv('Obesity.csv')
df.drop_duplicates(inplace=True)

colunas_ruidosas = ['FCVC', 'NCP', 'CH2O', 'FAF', 'TUE']
df[colunas_ruidosas] = df[colunas_ruidosas].round().astype(int)

colunas_texto = df.select_dtypes(include=['object']).columns
for col in colunas_texto:
    df[col] = df[col].str.strip()

print(f"Dataset carregado com {df.shape[0]} linhas e {df.shape[1]} colunas.")

Dataset carregado com 2087 linhas e 17 colunas.


In [2]:
if 'Weight' in df.columns and 'Height' in df.columns:
    df['BMI'] = df['Weight'] / (df['Height'] ** 2)

colunas_binarias = ['family_history', 'FAVC', 'SMOKE', 'SCC']
for col in colunas_binarias:
    if col in df.columns and df[col].dtype == 'object':
        df[col] = df[col].map({'no': 0, 'yes': 1})

mapeamento_freq = {'no': 0, 'Sometimes': 1, 'Frequently': 2, 'Always': 3}
for col in ['CAEC', 'CALC']:
    if col in df.columns and df[col].dtype == 'object':
        df[col] = df[col].map(mapeamento_freq)

print("Feature Engineering aplicado com sucesso.")

Feature Engineering aplicado com sucesso.


In [3]:
TARGET = 'Obesity'
X = df.drop(columns=[TARGET])
y = df[TARGET]

cat_cols = X.select_dtypes(include=['object']).columns.tolist()
num_cols = [c for c in X.columns if c not in cat_cols]

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', drop='first'))
])

preprocess = ColumnTransformer(transformers=[
    ('num', numeric_transformer, num_cols),
    ('cat', categorical_transformer, cat_cols)
])

modelos = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1, class_weight='balanced_subsample'),
    "Gradient Boosting": GradientBoostingClassifier(n_estimators=100, random_state=42),
    "KNN": KNeighborsClassifier(n_neighbors=5),
    "SVC": SVC(random_state=42)
}

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print("Comparação entre os Modelos...\n")

resultados = []
melhor_acuracia = 0
clf_campeao = None

for nome, modelo in modelos.items():
    clf_temp = Pipeline([('preprocess', preprocess), ('model', modelo)])
    clf_temp.fit(X_train, y_train)
    pred = clf_temp.predict(X_test)
    acc = accuracy_score(y_test, pred)

    resultados.append({'Modelo': nome, 'Acurácia (%)': acc * 100})

    if acc > melhor_acuracia:
        melhor_acuracia = acc
        clf_campeao = clf_temp

ranking_df = pd.DataFrame(resultados).sort_values(by='Acurácia (%)', ascending=False).reset_index(drop=True)
print("🏆 RANKING DE PERFORMANCE:")
print(ranking_df.to_string(index=False))
print("-" * 50)

ordem_obesidade = [
    'Insufficient_Weight', 'Normal_Weight', 'Overweight_Level_I',
    'Overweight_Level_II', 'Obesity_Type_I', 'Obesity_Type_II', 'Obesity_Type_III'
]

pred_campeao = clf_campeao.predict(X_test)
nome_campeao = ranking_df.iloc[0]['Modelo']
print(f"\nDetalhes do Modelo Vencedor ({nome_campeao}):")
print(classification_report(y_test, pred_campeao, labels=ordem_obesidade))

Comparação entre os Modelos...

🏆 RANKING DE PERFORMANCE:
             Modelo  Acurácia (%)
      Random Forest     98.564593
  Gradient Boosting     98.086124
      Decision Tree     97.846890
Logistic Regression     91.626794
                SVC     89.712919
                KNN     77.511962
--------------------------------------------------

Detalhes do Modelo Vencedor (Random Forest):
                     precision    recall  f1-score   support

Insufficient_Weight       1.00      1.00      1.00        53
      Normal_Weight       0.95      1.00      0.97        57
 Overweight_Level_I       1.00      0.96      0.98        55
Overweight_Level_II       0.98      0.95      0.96        58
     Obesity_Type_I       0.99      1.00      0.99        70
    Obesity_Type_II       0.98      1.00      0.99        60
   Obesity_Type_III       1.00      0.98      0.99        65

           accuracy                           0.99       418
          macro avg       0.99      0.99      0.99      

In [4]:
# EXPORTAÇÃO PARA O STREAMLIT

# Gerando o arquivo com nome .joblib
joblib.dump(clf_campeao, 'modelo_obesidade.joblib', compress=3)

# Gerando o mesmo arquivo com nome .pkl
joblib.dump(clf_campeao, 'modelo_obesidade.pkl', compress=3)

print("O motor de Inteligência Artificial foi exportado em DOIS formatos.")

O motor de Inteligência Artificial foi exportado em DOIS formatos.
